### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:

import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")

model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1087ed400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1087ee120>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie"),
    year:int=Field(description="This year the movie was released"),
    director:str=Field(description="This is the director of the movie"),
    rating:float=Field(description="The movies rating out of 10")


In [3]:
model_with_structured_output=model.with_structured_output(Movie)

/Users/sanskarvishwakarma/Downloads/projects/AgenticAI/krish_naik_course_agentic_ai/.venv/lib/python3.13/site-packages/pydantic/json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/sanskarvishwakarma/Downloads/projects/AgenticAI/krish_naik_course_agentic_ai/.venv/lib/python3.13/site-packages/pydantic/json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='This year the movie was released'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/sanskarvishwakarma/Downloads/projects/AgenticAI/krish_naik_course_agentic_ai/.venv/lib/python3.13/site-packages/pydantic/json_schema.py:2463: Pydanti

In [5]:
response=model_with_structured_output.invoke("Provide the details about the movie Inception")

response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [7]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A Movie with details"""
    title:str=Field(description="The title of the movie"),
    year:int=Field(description="This year the movie was released"),
    director:str=Field(description="This is the director of the movie"),
    rating:float=Field(description="The movies rating out of 10")


model_with_structured_output=model.with_structured_output(Movie,include_raw=True)

response=model_with_structured_output.invoke("Provide the details about the movie Inception")

response

/Users/sanskarvishwakarma/Downloads/projects/AgenticAI/krish_naik_course_agentic_ai/.venv/lib/python3.13/site-packages/pydantic/json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/sanskarvishwakarma/Downloads/projects/AgenticAI/krish_naik_course_agentic_ai/.venv/lib/python3.13/site-packages/pydantic/json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='This year the movie was released'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/sanskarvishwakarma/Downloads/projects/AgenticAI/krish_naik_course_agentic_ai/.venv/lib/python3.13/site-packages/pydantic/json_schema.py:2463: Pydanti

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see what tools I have available. There\'s a Movie function that requires a rating, and optionally director, title, and year. The user didn\'t specify the rating, but the function requires it. I need to figure out the rating for Inception. I remember that Inception has a high rating on IMDb, maybe around 8.8. The director is Christopher Nolan, and the release year was 2010. Let me confirm those details. Yeah, that\'s right. So I\'ll use the Movie function with the title "Inception", director "Christopher Nolan", year 2010, and rating 8.8. That should cover all the required parameters and provide the user with the necessary information.\n', 'tool_calls': [{'id': 'wxrds323h', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_us

### Nested Structure

In [10]:
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor] # Nested
    genres:list[str]
    budget:float | None=Field(None,description="Budget in million USD")

model_with_structured_output=model.with_structured_output(MovieDetails)

response=model_with_structured_output.invoke("Provide the details about the movie Inception")

response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Professor Fujita')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

### TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [11]:
from typing_extensions import TypedDict,Annotated

In [12]:
class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [13]:
model_with_structured_output=model.with_structured_output(MovieDict)

response=model_with_structured_output.invoke("Provide the details about the movie Inception")

response

{'director': 'Christopher Nolan',
 'rating': 8.8,
 'title': 'Inception',
 'year': 2010}

In [16]:
from typing_extensions import TypedDict,Annotated

class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor] # Nested
    genres:list[str]
    budget:float | None=Field(None,description="Budget in million USD")

model_with_structured_output=model.with_structured_output(MovieDetails)

response=model_with_structured_output.invoke("Provide the details about the movie Inception")

response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Jacob'}],
 'genres': ['Science Fiction', 'Action', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [18]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### Data Classes

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [20]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='9c934628-e907-4bec-a9b6-a4fd1c992fbb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". \n\nFirst, I need to identify the different parts of the contact info. The name is John Doe. The email is john@example.com. The phone number is (555) 123-4567. \n\nI should check if there\'s a specific format required for the phone number. The example shows it with parentheses and a space, but maybe the function expects a different format. However, the function\'s parameters just mention a string type for the phone, so I can probably leave it as is.\n\nNext, I need to map these parts to the required parameters of the ContactInfo function: name, email, and phone. All three are required. The order in

In [22]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [23]:
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str 
    email: str 
    phone: str 

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='85420443-86c7-47b5-9fe5-31c9d076d56f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". Let me look at the tools provided. There\'s a function called ContactInfo with parameters name, email, and phone. All three are required.\n\nFirst, I need to parse the input. The name is John Doe. The email is clearly john@example.com. The phone number is (555) 123-4567. I should check if the phone number is in the correct format. The function expects a string for each parameter. So I\'ll map each part to the respective parameter. Name is straightforward. Email is valid. Phone number might need to have the parentheses and hyphen removed, but the function\'s parameters are strings, so maybe it\'s okay to leave 